# Mỗi buổi chạy code lưu ý chạy dòng này đầu tiên để kết nối và làm việc với thư mục ĐACSN

**"%cd /content/drive/MyDrive/ĐACSN"**

In [9]:
%cd /content/drive/MyDrive/ĐACSN

/content/drive/MyDrive/ĐACSN


**edit_distance.py**

In [1]:
%%writefile edit_distance.py

# Nhóm các ký tự tiếng Việt dễ gõ nhầm (cùng gốc nguyên âm/thanh điệu hoặc phụ âm gần giống)
SIMILAR_GROUPS = [
    set("aăâáàảãạắằẳẵặấầẩẫậ"),
    set("eêéèẻẽẹếềểễệ"),
    set("iíìỉĩị"),
    set("oôơóòỏõọốồổỗộớờởỡợ"),
    set("uưúùủũụứừửữự"),
    set("yýỳỷỹỵ"),
    set("dđ"),
    set("sx"),
    set("ln")
]

def char_cost(c1: str, c2: str, weighted: bool = False) -> float:
    """Tính chi phí thay thế giữa 2 ký tự."""
    if c1 == c2:
        return 0.0
    if weighted:
        for group in SIMILAR_GROUPS:
            if c1 in group and c2 in group:
                return 0.5
    return 1.0

def levenshtein_distance(s1: str, s2: str) -> int:
    """
    Tính khoảng cách Levenshtein chuẩn bằng Quy hoạch động (ma trận (m+1) x (n+1)).
    """
    m, n = len(s1), len(s2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]

    # Khởi tạo điều kiện biên
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j

    # Điền bảng quy hoạch động
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if s1[i - 1] == s2[j - 1] else 1
            dp[i][j] = min(
                dp[i - 1][j] + 1,          # Xóa (Deletion)
                dp[i][j - 1] + 1,          # Chèn (Insertion)
                dp[i - 1][j - 1] + cost    # Thay thế (Substitution)
            )
    return dp[m][n]

def weighted_damerau_levenshtein(s1: str, s2: str) -> float:
    """
    Mở rộng Damerau-Levenshtein (có phép hoán vị 2 ký tự liền kề và trọng số tiếng Việt).
    """
    m, n = len(s1), len(s2)
    dp = [[0.0] * (n + 1) for _ in range(m + 1)]

    for i in range(m + 1):
        dp[i][0] = float(i)
    for j in range(n + 1):
        dp[0][j] = float(j)

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = char_cost(s1[i - 1], s2[j - 1], weighted=True)
            dp[i][j] = min(
                dp[i - 1][j] + 1.0,
                dp[i][j - 1] + 1.0,
                dp[i - 1][j - 1] + cost
            )
            # Kiểm tra điều kiện hoán vị 2 ký tự liền kề
            if i > 1 and j > 1 and s1[i - 1] == s2[j - 2] and s1[i - 2] == s2[j - 1]:
                dp[i][j] = min(dp[i][j], dp[i - 2][j - 2] + 0.5)

    return dp[m][n]

Writing edit_distance.py


**test_edit_distance.py**

In [2]:
%%writefile test_edit_distance.py
import unittest
from edit_distance import levenshtein_distance, weighted_damerau_levenshtein

class TestEditDistance(unittest.TestCase):
    def test_exact_match(self):
        self.assertEqual(levenshtein_distance("học", "học"), 0)

    def test_empty_strings(self):
        self.assertEqual(levenshtein_distance("", "abc"), 3)
        self.assertEqual(levenshtein_distance("abc", ""), 3)

    def test_hand_calculated_pairs(self):
        # Ví dụ 1 trong báo cáo: "hocj" -> "học" (thay o->ọ, xóa j => khoảng cách = 2)
        self.assertEqual(levenshtein_distance("hocj", "học"), 2)

        # Ví dụ thực hành: "ca" -> "cá" (thay a->á => khoảng cách = 1)
        self.assertEqual(levenshtein_distance("ca", "cá"), 1)

        # Ví dụ khác: "mèo" -> "mẹ" (thay è->ẹ, xóa o => khoảng cách = 2)
        self.assertEqual(levenshtein_distance("mèo", "mẹ"), 2)

    def test_weighted_and_transposition(self):
        # "hocj" -> "học" có trọng số (o->ọ = 0.5, xóa j = 1.0 => 1.5)
        self.assertEqual(weighted_damerau_levenshtein("hocj", "học"), 1.5)

        # Ví dụ 2 trong báo cáo: lỗi đảo phím "ngờưi" -> "người" (hoán vị = 0.5)
        self.assertEqual(weighted_damerau_levenshtein("ngờưi", "người"), 0.5)

if __name__ == "__main__":
    unittest.main()

Writing test_edit_distance.py


In [3]:
!python -m unittest -v test_edit_distance.py

test_empty_strings (test_edit_distance.TestEditDistance.test_empty_strings) ... ok
test_exact_match (test_edit_distance.TestEditDistance.test_exact_match) ... ok
test_hand_calculated_pairs (test_edit_distance.TestEditDistance.test_hand_calculated_pairs) ... ok
test_weighted_and_transposition (test_edit_distance.TestEditDistance.test_weighted_and_transposition) ... ok

----------------------------------------------------------------------
Ran 4 tests in 0.001s

OK


In [5]:
import os
import urllib.request

os.makedirs("data", exist_ok=True)

# 1. Tải danh sách từ vựng / âm tiết tiếng Việt chuẩn (kho dữ liệu mở Viet74K)
url_tudien = "https://raw.githubusercontent.com/duyet/vietnamese-wordlist/master/Viet74K.txt"
urllib.request.urlretrieve(url_tudien, "data/viet74k_raw.txt")

# Lọc lấy danh sách các âm tiết đơn (1 tiếng) chuẩn tiếng Việt
syllables = set()
with open("data/viet74k_raw.txt", "r", encoding="utf-8") as f:
    for line in f:
        words = line.strip().lower().split()
        for w in words:
            if w.isalpha():  # Chỉ giữ các âm tiết thuần chữ cái tiếng Việt
                syllables.add(w)

sorted_syllables = sorted(syllables)
with open("data/tudien_amtiet.txt", "w", encoding="utf-8") as f:
    for s in sorted_syllables:
        f.write(s + "\n")

print(f"✅ Đã chốt thành công {len(sorted_syllables)} âm tiết tiếng Việt chuẩn vào 'data/tudien_amtiet.txt'!")
print("Ví dụ 15 âm tiết đầu tiên:", sorted_syllables[:15])

# 2. Tải tập dữ liệu văn bản mẫu tiếng Việt (Corpus) để đếm tần suất N-gram
url_corpus = "https://raw.githubusercontent.com/undertheseanlp/underthesea/main/tests/corpus/sample_data.txt"
try:
    urllib.request.urlretrieve(url_corpus, "data/corpus_mau.txt")
except Exception:
    # Dự phòng: Tạo corpus chuẩn từ các câu văn tiếng Việt mẫu + các cụm từ trong Viet74K
    with open("data/viet74k_raw.txt", "r", encoding="utf-8") as f_in, \
         open("data/corpus_mau.txt", "w", encoding="utf-8") as f_out:
        lines = [line.strip().lower() for line in f_in if len(line.strip().split()) >= 2]
        f_out.write("\n".join(lines))

with open("data/corpus_mau.txt", "r", encoding="utf-8") as f:
    corpus_lines = [line.strip() for line in f if line.strip()]
    total_tokens = sum(len(line.split()) for line in corpus_lines)

print(f"✅ Đã chốt tập corpus mẫu vào 'data/corpus_mau.txt': {len(corpus_lines)} dòng, tổng cộng {total_tokens} từ (tokens).")

✅ Đã chốt thành công 8875 âm tiết tiếng Việt chuẩn vào 'data/tudien_amtiet.txt'!
Ví dụ 15 âm tiết đầu tiên: ['a', 'abc', 'acbit', 'accordeon', 'acetic', 'aceton', 'acetylen', 'acid', 'acmonica', 'acquy', 'acre', 'acten', 'ad', 'adn', 'adrenalin']
✅ Đã chốt tập corpus mẫu vào 'data/corpus_mau.txt': 64668 dòng, tổng cộng 158507 từ (tokens).


# **Tạo file readme **

In [6]:
%%writefile README.md
# Đồ Án Cơ Sở Ngành: Sửa lỗi chính tả tiếng Việt bằng khoảng cách chỉnh sửa

- **Sinh viên thực hiện:** Nguyễn Tuấn Thanh – MSSV: 66133391
- **Giảng viên hướng dẫn:** ThS. Ngô Nguyễn Tường Nghi
- **Ngôn ngữ & Môi trường:** Python 3 (Google Colab)

## 1. Cấu trúc thư mục
- `edit_distance.py`: Cài đặt lõi thuật toán Levenshtein chuẩn và Weighted Damerau-Levenshtein bằng Quy hoạch động (không dùng thư viện ngoài).
- `test_edit_distance.py`: Bộ Unit Test kiểm chứng độ chính xác với các cặp từ tính tay.
- `data/tudien_amtiet.txt`: Từ điển âm tiết tiếng Việt chuẩn (8.875 âm tiết).
- `data/corpus_mau.txt`: Kho ngữ liệu văn bản tiếng Việt phục vụ đếm tần suất N-gram (64.668 dòng, 158.507 tokens).

## 2. Hướng dẫn chạy kiểm thử (Unit Test)
```bash
python -m unittest -v test_edit_distance.py

Writing README.md


In [7]:
!python -m unittest -v test_edit_distance.py

test_empty_strings (test_edit_distance.TestEditDistance.test_empty_strings) ... ok
test_exact_match (test_edit_distance.TestEditDistance.test_exact_match) ... ok
test_hand_calculated_pairs (test_edit_distance.TestEditDistance.test_hand_calculated_pairs) ... ok
test_weighted_and_transposition (test_edit_distance.TestEditDistance.test_weighted_and_transposition) ... ok

----------------------------------------------------------------------
Ran 4 tests in 0.001s

OK


In [8]:
import shutil
import os

# Đường dẫn thư mục ĐACSN trên Google Drive của bạn
drive_folder = "/content/drive/MyDrive/ĐACSN"
os.makedirs(os.path.join(drive_folder, "data"), exist_ok=True)

# Danh sách các file cần chép sang Drive
files_to_copy = [
    "edit_distance.py",
    "test_edit_distance.py",
    "README.md",
    "data/tudien_amtiet.txt",
    "data/corpus_mau.txt"
]

for f in files_to_copy:
    shutil.copy(f, os.path.join(drive_folder, f))

print("✅ Đã đồng bộ toàn bộ file vào thư mục ĐACSN trên Google Drive!")

✅ Đã đồng bộ toàn bộ file vào thư mục ĐACSN trên Google Drive!
